In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
TARGET = "Target_local_recurrence"
pd.set_option("display.max_columns", None)
print(f"numpy {np.__version__}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}")

numpy 2.5.3; pandas 3.0.6; scikit-learn 1.9.1


In [2]:
analytic = pd.read_csv("../w4/analytic_table.csv")
test_membership = pd.read_csv("../w5/test.csv", usecols=["Patient_ID", TARGET])
previous_train_ids = pd.read_csv("../w5/train.csv", usecols=["Patient_ID"])["Patient_ID"]
assert analytic.Patient_ID.is_unique and test_membership.Patient_ID.is_unique
assert set(previous_train_ids).isdisjoint(test_membership.Patient_ID)
assert set(previous_train_ids) | set(test_membership.Patient_ID) == set(analytic.Patient_ID)
assert analytic.set_index("Patient_ID").loc[test_membership.Patient_ID, TARGET].tolist() == test_membership[TARGET].tolist()

pool = analytic.loc[analytic.Patient_ID.isin(previous_train_ids)].copy()
train, validation = train_test_split(pool, test_size=0.2, stratify=pool[TARGET], random_state=SEED)
train = train.sort_values("Patient_ID").reset_index(drop=True)
validation = validation.sort_values("Patient_ID").reset_index(drop=True)
assert set(train.Patient_ID).isdisjoint(validation.Patient_ID)
assert set(train.Patient_ID) | set(validation.Patient_ID) == set(previous_train_ids)
repeat_train, repeat_validation = train_test_split(pool, test_size=0.2, stratify=pool[TARGET], random_state=SEED)
assert set(repeat_train.Patient_ID) == set(train.Patient_ID)
assert set(repeat_validation.Patient_ID) == set(validation.Patient_ID)

samples = {"Обучение": train, "Валидация": validation, "Тест (отложен)": test_membership}
display(pd.DataFrame({name: {"Пациентов": len(frame), "Исход 1": int(frame[TARGET].sum())}
                      for name, frame in samples.items()}).T)
for name, frame in samples.items():
    print(f"{name}: {sorted(frame.Patient_ID.tolist())}")

,Пациентов,Исход 1
Обучение,84,7
Валидация,21,2
Тест (отложен),35,3


Обучение: [1, 7, 8, 9, 10, 11, 13, 15, 19, 20, 22, 23, 25, 26, 27, 32, 35, 37, 42, 46, 49, 51, 52, 54, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 72, 73, 74, 75, 78, 79, 83, 84, 86, 87, 91, 93, 94, 97, 98, 101, 102, 104, 105, 108, 109, 111, 115, 116, 117, 118, 121, 122, 124, 125, 131, 132, 133, 137, 138, 144, 146, 150, 159, 169, 178, 179, 194, 196, 276, 285, 293, 302, 312]
Валидация: [5, 6, 12, 24, 38, 41, 43, 45, 56, 64, 89, 90, 103, 106, 120, 123, 141, 143, 154, 158, 273]
Тест (отложен): [2, 4, 21, 29, 31, 34, 39, 44, 47, 48, 55, 69, 70, 77, 80, 82, 85, 95, 99, 100, 107, 113, 114, 119, 128, 135, 136, 139, 140, 142, 152, 156, 170, 174, 313]


In [3]:
binary = ["Gender", "Induction_Chemotherapy", "Concurrent_chemotherapy"]
ordinal = ["T_category", "N_category", "AJCC_Stage", "Pathological_grade"]
nominal = ["Race", "Tumor_side", "Tumor_subsite", "Smoking_status_at_diagnosis"]
quantitative = [col for col in pool if col not in ["Patient_ID", TARGET] + binary + ordinal + nominal]
features = quantitative + ordinal + binary + nominal
X_train, y_train = train[features], train[TARGET]
X_validation, y_validation = validation[features], validation[TARGET]
assert len(features) == 31 and len(quantitative) == 20

ordinal_codes = {
    "T_category": {1: 1, 2: 2, 3: 3, 4: 4},
    "N_category": {"0": 0, "1": 1, "2a": 2, "2b": 3, "2c": 4, "3": 5},
    "AJCC_Stage": {"I": 1, "II": 2, "III": 3, "IV": 4},
    "Pathological_grade": {"I": 1, "I-II": 1.5, "II": 2, "II-III": 2.5, "III": 3, "IV": 4},
}
binary_codes = {"Gender": {"Female": 0, "Male": 1},
                "Induction_Chemotherapy": {"N": 0, "Y": 1},
                "Concurrent_chemotherapy": {"N": 0, "Y": 1}}

def encode_ordinal(frame):
    return pd.DataFrame({col: frame[col].map(ordinal_codes[col]) for col in frame}, index=frame.index)

def encode_binary(frame):
    return pd.DataFrame({col: frame[col].map(binary_codes[col]) for col in frame}, index=frame.index)

def grade_missing(frame):
    return frame.isna().astype(float)

numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
preparation = ColumnTransformer([
    ("quantitative", clone(numeric), quantitative),
    ("ordinal", Pipeline([
        ("encode", FunctionTransformer(encode_ordinal, feature_names_out="one-to-one")),
        ("numeric", clone(numeric)),
    ]), ordinal),
    ("binary", Pipeline([
        ("encode", FunctionTransformer(encode_binary, feature_names_out="one-to-one")),
        ("impute", SimpleImputer(strategy="most_frequent")),
    ]), binary),
    ("nominal", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(min_frequency=5, drop="first", handle_unknown="infrequent_if_exist", sparse_output=False)),
    ]), nominal),
    ("grade_missing", FunctionTransformer(grade_missing, feature_names_out="one-to-one"), ["Pathological_grade"]),
], remainder="drop")

display(pd.DataFrame({"Группа": ["Количественные", "Порядковые", "Бинарные", "Номинальные"],
                      "Число исходных признаков": [len(quantitative), len(ordinal), len(binary), len(nominal)]}))

,Группа,Число исходных признаков
0,Количественные,20
1,Порядковые,4
2,Бинарные,3
3,Номинальные,4


In [4]:
classifiers = {
    "Логистическая регрессия": LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000, random_state=SEED),
    "Дерево решений": DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, class_weight="balanced", random_state=SEED),
    "Случайный лес": RandomForestClassifier(n_estimators=300, max_depth=3, min_samples_leaf=5,
                                             max_features="sqrt", class_weight="balanced_subsample",
                                             random_state=SEED, n_jobs=2),
}
parameter_names = ["C", "n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight", "random_state"]
display(pd.DataFrame({name: {key: model.get_params().get(key, "—") for key in parameter_names}
                      for name, model in classifiers.items()}).T)
models = {}
rows = []
for name, classifier in classifiers.items():
    model = Pipeline([("prepare", clone(preparation)), ("classifier", clone(classifier))])
    model.fit(X_train, y_train)
    models[name] = model
    fitted = model.named_steps["prepare"]
    q = fitted.named_transformers_["quantitative"]
    assert np.allclose(q.named_steps["impute"].statistics_, X_train[quantitative].median().to_numpy())
    assert int(q.named_steps["scale"].n_samples_seen_) == len(train)
    transformed = fitted.transform(X_validation)
    assert np.isfinite(transformed).all()
    rows.append({"Модель": name, "Обучающих пациентов": len(train), "Подготовленных признаков": transformed.shape[1]})
display(pd.DataFrame(rows).set_index("Модель"))

,C,n_estimators,max_depth,min_samples_leaf,max_features,class_weight,random_state
Логистическая регрессия,0.1,—,—,—,—,balanced,42
Дерево решений,—,—,3,8,None,balanced,42
Случайный лес,—,300,3,5,sqrt,balanced_subsample,42


,Обучающих пациентов,Подготовленных признаков
Модель,,
Логистическая регрессия,84,36
Дерево решений,84,36
Случайный лес,84,36


In [5]:
baseline = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(train), 1)), y_train)
train_predictions = {name: model.predict(X_train) for name, model in models.items()}
validation_predictions = {name: model.predict(X_validation) for name, model in models.items()}
train_predictions["Всегда нет рецидива (ориентир)"] = baseline.predict(np.zeros((len(train), 1)))
validation_predictions["Всегда нет рецидива (ориентир)"] = baseline.predict(np.zeros((len(validation), 1)))

comparison = pd.DataFrame([
    {"Модель": name,
     "Accuracy: обучение": accuracy_score(y_train, train_predictions[name]),
     "Accuracy: валидация": accuracy_score(y_validation, validation_predictions[name]),
     "Balanced accuracy: валидация": balanced_accuracy_score(y_validation, validation_predictions[name]),
     "Чувствительность: валидация": recall_score(y_validation, validation_predictions[name], zero_division=0)}
    for name in validation_predictions
]).set_index("Модель")
display(comparison.round(4))

,Accuracy: обучение,Accuracy: валидация,Balanced accuracy: валидация,Чувствительность: валидация
Модель,,,,
Логистическая регрессия,0.8929,0.8095,0.4474,0.0
Дерево решений,0.9524,0.8095,0.4474,0.0
Случайный лес,1.0000,0.9048,0.5000,0.0
Всегда нет рецидива (ориентир),0.9167,0.9048,0.5000,0.0


In [6]:
confusions = []
for name, prediction in validation_predictions.items():
    tn, fp, fn, tp = confusion_matrix(y_validation, prediction, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(validation)
    confusions.append({"Модель": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
display(pd.DataFrame(confusions).set_index("Модель"))
print("Тест: только состав и число исходов; подготовка, обучение и предсказания не выполнялись.")

,TN,FP,FN,TP
Модель,,,,
Логистическая регрессия,17,2,2,0
Дерево решений,17,2,2,0
Случайный лес,19,0,2,0
Всегда нет рецидива (ориентир),19,0,2,0


Тест: только состав и число исходов; подготовка, обучение и предсказания не выполнялись.
